# Neo4j Векторный поиск (Encounter сем близ + фильтры ТОЛЬКО scope + семант близость триплетов)
# Проблема: выдаем top-k, неясно сколько именно
1. Создание эмбедингов для Encounter через карточки
2. Поиск Encounters, наиболее близких к вопросу
3. LLM ищет scope фильтры из списка
4. Фильтрация encounters по фильтрам scope
5. Добыча триплетов на глубине 2 от Encounter
6. Создание эмбедингов для триплетов
7. Фильтрация триплетов семантически
8. Подсчет метрик
9. Запись выдачи на диск


# 1. Создание эмбедингов для Encounter через карточки

In [108]:
# 1 создание эмбедингов для Encounters через карточки + инфа о связи bva-paris и аналоги
# Подключение к БД
from neo4j import GraphDatabase

NEO4J_URI = "neo4j://localhost:7689"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"
NEO4J_DB = "neo4j"

EMBEDDING_PROP = "embedding_e5"
VECTOR_INDEX_NAME = "encounter_embedding_e5"
VECTOR_DIM = 768  # e5-base-v2

def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Создание векторного индекса по свойству Encounter.embedding_e5 
def ensure_vector_index(driver):
    with driver.session(database=NEO4J_DB) as session:
        session.run(
            f"""
            CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
            FOR (e:Encounter) ON (e.{EMBEDDING_PROP})
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {VECTOR_DIM}, `vector.similarity_function`: 'cosine' }} }}
            """
        ).consume()
from typing import Dict, List

# Map smaller/metro airports to their larger city for embedding context
AIRPORT_ALIAS_CITY = {
    "airport_bva": "Paris",
    "airport_cdg": "Paris",
    "airport_ory": "Paris",
    "airport_sdg": "Paris",
    "airport_memmingen": "Munich",
    "airport_malpensa": "Milan",
    "airport_marco_polo": "Venice",
    "airport_fiumicino": "Rome",
    "airport_tegel": "Berlin",
    "airport_schonefeld": "Berlin",
    "airport_pisa": "Florence",
}

def fetch_encounter_cards(driver) -> Dict[str, str]:
    query = '''
    MATCH (e:Encounter)
    OPTIONAL MATCH (e)-[:atCity]->(city:City)
    OPTIONAL MATCH (e)-[:atAirport]->(airport:Airport)
    OPTIONAL MATCH (e)-[:atCountry]->(country:Country)
    OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
    RETURN e.key AS ekey,
           collect(DISTINCT city.key) AS cities,
           collect(DISTINCT airport.key) AS airports,
           collect(DISTINCT country.key) AS countries,
           collect(DISTINCT di.documentType) AS doc_types,
           collect(DISTINCT q.topicKey) AS topics
    '''
    cards = {}
    with driver.session(database=NEO4J_DB) as session:
        for r in session.run(query):
            ekey = r["ekey"]
            if not ekey:
                continue
            parts = [f"Encounter: {ekey}"]
            if r["cities"]:
                parts.append("Cities: " + ", ".join(sorted([c for c in r["cities"] if c])))
            if r["airports"]:
                airports = sorted([a for a in r["airports"] if a])
                parts.append("Airports: " + ", ".join(airports))
                alias_cities = sorted({AIRPORT_ALIAS_CITY[a] for a in airports if a in AIRPORT_ALIAS_CITY})
                if alias_cities:
                    parts.append("AliasCities: " + ", ".join(alias_cities))
            if r["countries"]:
                parts.append("Countries: " + ", ".join(sorted([c for c in r["countries"] if c])))
            if r["doc_types"]:
                parts.append("Documents: " + ", ".join(sorted([d for d in r["doc_types"] if d])))
            if r["topics"]:
                parts.append("Topics: " + ", ".join(sorted([t for t in r["topics"] if t])))
            cards[ekey] = " | ".join(parts)
    return cards

from typing import Iterable
try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError("Install sentence-transformers to compute E5 embeddings") from e

def embed_texts_e5(model: SentenceTransformer, texts: Iterable[str], is_query: bool = False) -> List[List[float]]:
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + t for t in texts]
    vecs = model.encode(texts, normalize_embeddings=True)
    return [v.tolist() for v in vecs]

def build_embeddings_for_encounters(cards: Dict[str, str]):
    model = SentenceTransformer("intfloat/e5-base-v2")
    keys = list(cards.keys())
    texts = [cards[k] for k in keys]
    vectors = embed_texts_e5(model, texts, is_query=False)
    return keys, vectors
def write_embeddings(driver, keys: List[str], vectors: List[List[float]]):
    with driver.session(database=NEO4J_DB) as session:
        for k, v in zip(keys, vectors):
            session.run(
                "MATCH (e:Encounter {key:$k}) SET e.%s = $v" % EMBEDDING_PROP,
                k=k, v=v
            ).consume()
# ___________________________________________________________________________________________________________________
# ПАЙПЛАЙН создания и записи всех ембедингов для Encounters
with get_driver() as driver:
    ensure_vector_index(driver)
    cards = fetch_encounter_cards(driver)
    keys, vecs = build_embeddings_for_encounters(cards)
    write_embeddings(driver, keys, vecs)
    print(f"Wrote embeddings for {len(keys)} encounters")


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7304.14it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wrote embeddings for 469 encounters


# 2. Поиск Encounters, наиболее близких к вопросу

In [109]:
# 2 Поиск top-k Encounter'ов наиболее близких семантически к данному вопросу

QUESTION = "Which documents do I need to bring to passport control in Paris?"
TOP_K = 157

# Build query embedding and retrieve nearest Encounters
with get_driver() as driver:
    # reuse existing model if present, otherwise load
    try:
        model
    except NameError:
        model = SentenceTransformer("intfloat/e5-base-v2")
    q_vec = embed_texts_e5(model, [QUESTION], is_query=True)[0]

    cypher = """
    CALL db.index.vector.queryNodes($index, $k, $vector)
    YIELD node, score
    WHERE 'Encounter' IN labels(node)
    RETURN coalesce(node.key, elementId(node)) AS encounter_key, score
    ORDER BY score DESC
    """
    rows = []
    with driver.session(database=NEO4J_DB) as session:
        rows = session.run(cypher, index=VECTOR_INDEX_NAME, k=TOP_K, vector=q_vec).data()

top_encounter_keys = [r['encounter_key'] for r in rows]
print(f"Top-{TOP_K} encounters: {len(top_encounter_keys)}")
print(top_encounter_keys[:10])


Top-157 encounters: 157
['encounter_12270454_companion1_1', 'encounter_12100144_author_1', 'encounter_11709466_author_1', 'encounter_12204974_author_1', 'encounter_11870493_author_1', 'encounter_12265196_author_1', 'encounter_3902451_friend1_1', 'encounter_3812199_author_1', 'encounter_11383255_author_1', 'encounter_9343311_author_1']


# 3. LLM ищет scope фильтры из списка

In [146]:
# 2b LLM scope filter (cities/airports/countries)
import os, json, requests

def fetch_scope_catalog(driver):
    with driver.session(database=NEO4J_DB) as session:
        cities = [r['k'] for r in session.run("MATCH (c:City) WHERE c.key IS NOT NULL RETURN DISTINCT c.key AS k ORDER BY k")]
        airports = [r['k'] for r in session.run("MATCH (a:Airport) WHERE a.key IS NOT NULL RETURN DISTINCT a.key AS k ORDER BY k")]
        countries = [r['k'] for r in session.run("MATCH (c:Country) WHERE c.key IS NOT NULL RETURN DISTINCT c.key AS k ORDER BY k")]
    return {'cities': cities, 'airports': airports, 'countries': countries}

def llm_generate_scope_filter(question, scope_catalog):
    %pip -q install python-dotenv

    from dotenv import load_dotenv
    import os

    load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

    api_key = os.getenv("DEEPSEEK_API_KEY")
    print("DEEPSEEK_API_KEY loaded:", bool(api_key))

    if not api_key:
        raise RuntimeError('DEEPSEEK_API_KEY is not set')

    prompt = f"""You are a filter generator.
Given a QUESTION and a catalog of available scope keys, select only the keys relevant to the question.
If no restriction is needed for a scope type, return an empty list for that type.
Important rule: empty list means ALL values are allowed; non-empty list means ONLY those values are allowed.
Return ONLY valid JSON with this exact shape:
{{
  "scope": {{
    "cities": [..],
    "airports": [..],
    "countries": [..]
  }}
}}

QUESTION:
{question}

SCOPE_CATALOG:
{json.dumps(scope_catalog)}

Return only JSON."""

    payload = {
        'model': 'deepseek-reasoner',
        'temperature': 0,
        'messages': [
            {'role': 'system', 'content': 'You output only JSON.'},
            {'role': 'user', 'content': prompt}
        ]
    }

    resp = requests.post(
        'https://api.deepseek.com/chat/completions',
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        data=json.dumps(payload)
    )
    resp.raise_for_status()
    content = resp.json()['choices'][0]['message']['content']
    return json.loads(content)

with get_driver() as driver:
    scope_catalog = fetch_scope_catalog(driver)

scope_filter = llm_generate_scope_filter(QUESTION, scope_catalog)
print(scope_filter)


Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True
{'scope': {'cities': [], 'airports': ['airport_cologne_bonn', 'airport_frankfurt', 'airport_memmingen', 'airport_munich', 'airport_schonefeld', 'airport_stuttgart', 'airport_tegel'], 'countries': ['country_germany']}}


# 4. Фильтрация encounters по фильтрам scope

In [111]:
# 2c filter encounters by scope
def filter_encounters_by_scope(driver, encounter_keys, scope_filter):
    if not encounter_keys:
        return []
    scope = scope_filter.get('scope', {})
    cities = set(scope.get('cities', []))
    airports = set(scope.get('airports', []))
    countries = set(scope.get('countries', []))

    with driver.session(database=NEO4J_DB) as session:
        rows = session.run("""
        MATCH (e:Encounter) WHERE e.key IN $keys
        OPTIONAL MATCH (e)-[:atCity]->(c:City)
        OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
        OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
        RETURN e.key AS key,
               collect(DISTINCT c.key) AS cities,
               collect(DISTINCT a.key) AS airports,
               collect(DISTINCT co.key) AS countries
        """, keys=encounter_keys).data()

    out = []
    for r in rows:
        cset = set([x for x in (r['cities'] or []) if x])
        aset = set([x for x in (r['airports'] or []) if x])
        coset = set([x for x in (r['countries'] or []) if x])

        city_ok = (not cities) or bool(cset & cities)
        airport_ok = (not airports) or bool(aset & airports)
        country_ok = (not countries) or bool(coset & countries)

        if city_ok and airport_ok and country_ok:
            out.append(r['key'])

    return out

with get_driver() as driver:
    scoped_encounter_keys = filter_encounters_by_scope(driver, top_encounter_keys, scope_filter)

print(f"Scoped encounters: {len(scoped_encounter_keys)}")
top_encounter_keys = scoped_encounter_keys


Scoped encounters: 67


# 5. Добыча триплетов на глубине 2 от Encounter

In [112]:
# 3 fetch triples within depth<=2 from the retrieved encounters

def fetch_triples_depth2(driver, encounter_keys):
    if not encounter_keys:
        return []
    triples = []
    q1 = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[r]->(o)
    RETURN labels(e)[0] AS s_label, coalesce(e.key, elementId(e)) AS s_key,
           type(r) AS p, labels(o)[0] AS o_label, coalesce(o.key, elementId(o)) AS o_key
    """
    q2 = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[]->(n)-[r]->(o)
    RETURN labels(n)[0] AS s_label, coalesce(n.key, elementId(n)) AS s_key,
           type(r) AS p, labels(o)[0] AS o_label, coalesce(o.key, elementId(o)) AS o_key
    """
    q_doc = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[:hasDocumentCheck]->(:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    WHERE di.documentType IS NOT NULL
    RETURN 'DocumentInstance' AS s_label, coalesce(di.key, elementId(di)) AS s_key,
           'documentType' AS p, 'Literal' AS o_label, toString(di.documentType) AS o_key
    """
    q_topic = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[:hasQuestioning]->(q:Questioning)
    WHERE q.topicKey IS NOT NULL
    RETURN 'Questioning' AS s_label, coalesce(q.key, elementId(q)) AS s_key,
           'topicKey' AS p, 'Literal' AS o_label, toString(q.topicKey) AS o_key
    """
    with driver.session(database=NEO4J_DB) as session:
        triples += session.run(q1, keys=encounter_keys).data()
        triples += session.run(q2, keys=encounter_keys).data()
        triples += session.run(q_doc, keys=encounter_keys).data()
        triples += session.run(q_topic, keys=encounter_keys).data()

    # de-duplicate
    seen = set()
    out = []
    for t in triples:
        key = (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out

with get_driver() as driver:
    triples_depth2 = fetch_triples_depth2(driver, top_encounter_keys)

print('Depth<=2 triples:', len(triples_depth2))


Depth<=2 triples: 802


# 6. Создание эмбедингов для триплетов

In [113]:
# 4a build embeddings for triples (simple approach)
# PLAN:
# 1) Convert each triple to a text string.
# 2) Embed all triple texts and the question.

def triple_to_text(t):
    return f"{t['s_label']} {t['s_key']} --{t['p']}--> {t['o_label']} {t['o_key']}"

triple_texts = [triple_to_text(t) for t in triples_depth2]

try:
    model
except NameError:
    model = SentenceTransformer('intfloat/e5-base-v2')

q_vec = embed_texts_e5(model, [QUESTION], is_query=True)[0]
t_vecs = embed_texts_e5(model, triple_texts, is_query=False)


# 7. Фильтрация триплетов семантически 

In [114]:
# 4b semantic filtering of triples
TOP_N_TRIPLES = 5000

import numpy as np
q = np.array(q_vec, dtype=np.float32)
T = np.array(t_vecs, dtype=np.float32)
q_norm = np.linalg.norm(q) + 1e-9
t_norm = np.linalg.norm(T, axis=1) + 1e-9
scores = (T @ q) / (t_norm * q_norm)

idx = np.argsort(-scores)[:min(TOP_N_TRIPLES, len(triples_depth2))]
triples_semantic = [triples_depth2[i] for i in idx]
print('Semantic triples:', len(triples_semantic))


Semantic triples: 802


# 8. Подсчет метрик

In [115]:
# 5 metrics: precision/recall/F1 vs gold for question 1
import json
from pathlib import Path

def normalize_triple(t):
    # supports both flat triples and nested {s:{label,key}, p, o:{label,key}}
    if 's_label' in t:
        return (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])
    # nested format
    s = t.get('s', {})
    o = t.get('o', {})
    return (s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key'))

def triples_to_set(triples):
    s = set()
    for t in triples:
        s.add(normalize_triple(t))
    return s

def load_q1_gold():
    base = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation/q_1')
    triples = []
    for folder in ['fr_1','fr_2','germany','italy','spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            triples.extend(data.get('triples', []))
    return triples

retrieved_set = triples_to_set(triples_semantic)
gold_set = triples_to_set(load_q1_gold())
print('retrieved', len(retrieved_set))
print('gold', len(gold_set))
tp = len(retrieved_set & gold_set)
precision = tp / len(retrieved_set) if retrieved_set else 0.0
recall = tp / len(gold_set) if gold_set else 0.0
f1 = (2*precision*recall/(precision+recall)) if (precision+recall)>0 else 0.0
print('precision', precision)
print('recall', recall)
print('f1', f1)
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


retrieved 802
gold 641
precision 0.7044887780548629
recall 0.8814352574102964
f1 0.7830907830907831


# 9. Запись выдачи на диск

In [116]:
# 6 save retrieved triples to disk
import json
from pathlib import Path

out_path = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/retrieved_triples_q_1.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(triples_semantic, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved', out_path)


Saved /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/retrieved_triples_q_1.json


In [148]:
# Batch pipeline: run multiple questions and write metrics (percent, integer)

import json

from pathlib import Path

import numpy as np

from sentence_transformers import SentenceTransformer



QUESTION_NUMBERS = [i for i in range(1, 87)]  # edit as needed

TOP_K = 157

TOP_N_TRIPLES = 5000

METRICS_PATH = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/metrics_current_question.csv')



def extract_question(qnum: int) -> str:

    p = Path(f'/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation_prompts/q_{qnum}_prompt.txt')

    if not p.exists():

        raise FileNotFoundError(p)

    lines = p.read_text(encoding='utf-8').splitlines()

    for i, line in enumerate(lines):

        if line.strip().upper() == 'QUESTION':

            for j in range(i+1, len(lines)):

                if lines[j].strip():

                    return lines[j].strip()

    raise RuntimeError(f'QUESTION not found in {p}')



def load_gold(qnum: int):

    base = Path(f'/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation/q_{qnum}')

    triples = []

    for folder in ['fr_1','fr_2','germany','italy','spain']:

        p = base / folder

        if not p.exists():

            continue

        for f in sorted(p.glob('annotation_*.json')):

            data = json.loads(f.read_text(encoding='utf-8'))

            triples.extend(data.get('triples', []))

    return triples



def normalize_triple(t):

    if 's_label' in t:

        return (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])

    s = t.get('s', {})

    o = t.get('o', {})

    return (s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key'))



def triples_to_set(triples):

    return {normalize_triple(t) for t in triples}



def triple_to_text(t):

    return f"{t['s_label']} {t['s_key']} --{t['p']}--> {t['o_label']} {t['o_key']}"



def update_metrics_csv(path: Path, qnum: int, precision: int, recall: int, f1: int):

    rows = {}

    if path.exists():

        for line in path.read_text(encoding='utf-8').splitlines()[1:]:

            if not line.strip():

                continue

            parts = line.split(',')

            if len(parts) >= 4:

                rows[int(parts[0])] = (parts[1], parts[2], parts[3])

    rows[qnum] = (str(precision), str(recall), str(f1))

    out_lines = ['question,precision,recall,f1']

    for q in sorted(rows):

        pr, rc, f1v = rows[q]

        out_lines.append(f'{q},{pr},{rc},{f1v}')

    path.write_text('\n'.join(out_lines) + '\n', encoding='utf-8')



# Ensure model exists

model = SentenceTransformer('intfloat/e5-base-v2')



for qnum in QUESTION_NUMBERS:

    question = extract_question(qnum)

    q_vec = embed_texts_e5(model, [question], is_query=True)[0]



    # semantic top-k encounters

    with get_driver() as driver:

        with driver.session(database=NEO4J_DB) as session:

            rows = session.run("""

            CALL db.index.vector.queryNodes($index, $k, $vector)

            YIELD node, score

            WHERE 'Encounter' IN labels(node)

            RETURN coalesce(node.key, elementId(node)) AS encounter_key, score

            ORDER BY score DESC

            """, index=VECTOR_INDEX_NAME, k=TOP_K, vector=q_vec).data()

    top_encounter_keys = [r['encounter_key'] for r in rows]



    # depth-2 triples

    triples_depth2 = fetch_triples_depth2(get_driver(), top_encounter_keys)



    # semantic filtering of triples

    triple_texts = [triple_to_text(t) for t in triples_depth2]

    t_vecs = embed_texts_e5(model, triple_texts, is_query=False)

    q = np.array(q_vec, dtype=np.float32)

    T = np.array(t_vecs, dtype=np.float32)

    q_norm = np.linalg.norm(q) + 1e-9

    t_norm = np.linalg.norm(T, axis=1) + 1e-9

    scores = (T @ q) / (t_norm * q_norm)

    idx = np.argsort(-scores)[:min(TOP_N_TRIPLES, len(triples_depth2))]

    triples_semantic = [triples_depth2[i] for i in idx]



    retrieved_set = triples_to_set(triples_semantic)

    gold_set = triples_to_set(load_gold(qnum))

    tp = len(retrieved_set & gold_set)

    precision = int(round((tp / len(retrieved_set)) * 100)) if retrieved_set else 0

    recall = int(round((tp / len(gold_set)) * 100)) if gold_set else 0

    f1 = int(round((2*precision*recall/(precision+recall)) if (precision+recall)>0 else 0))

    print(f'Q{qnum}: retrieved {len(retrieved_set)} gold {len(gold_set)} pr {precision} rc {recall} f1 {f1}')

    update_metrics_csv(METRICS_PATH, qnum, precision, recall, f1)



Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 11047.43it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q1: retrieved 1786 gold 641 pr 36 rc 100 f1 53
Q2: retrieved 1674 gold 108 pr 6 rc 100 f1 11
Q3: retrieved 1715 gold 109 pr 6 rc 100 f1 11
Q4: retrieved 1674 gold 39 pr 2 rc 100 f1 4
Q5: retrieved 1768 gold 23 pr 1 rc 100 f1 2
Q6: retrieved 1895 gold 430 pr 23 rc 100 f1 37
Q7: retrieved 1892 gold 347 pr 18 rc 100 f1 31
Q8: retrieved 1891 gold 293 pr 15 rc 100 f1 26
Q9: retrieved 1916 gold 304 pr 16 rc 100 f1 28
Q10: retrieved 1888 gold 104 pr 6 rc 100 f1 11
Q11: retrieved 1602 gold 195 pr 12 rc 100 f1 21
Q12: retrieved 1630 gold 53 pr 3 rc 100 f1 6
Q13: retrieved 1571 gold 106 pr 7 rc 100 f1 13
Q14: retrieved 1598 gold 26 pr 2 rc 100 f1 4
Q15: retrieved 1555 gold 19 pr 1 rc 100 f1 2
Q16: retrieved 1595 gold 56 pr 4 rc 100 f1 8
Q17: retrieved 1535 gold 26 pr 2 rc 100 f1 4
Q18: retrieved 1669 gold 22 pr 1 rc 100 f1 2
Q19: retrieved 1562 gold 9 pr 1 rc 100 f1 2
Q20: retrieved 1538 gold 6 pr 0 rc 100 f1 0
Q21: retrieved 1702 gold 1043 pr 61 rc 100 f1 76
Q22: retrieved 1952 gold 1889 pr 78 

In [144]:
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

# LLM выбирает фильтры из 3 листов (labels/predicates/keys) -> filter triples

In [272]:
# One-cell: choose filters from 3 lists (labels/predicates/keys) -> filter triples
import os, json, re, requests
from neo4j import GraphDatabase

QUESTION = "Which airport do people most often fly into in Germany?"
Q_NUM = 74 
LLM_MODEL = "gpt-5.4-nano"

# ---- Neo4j helpers ----

def run_cypher(query, params=None):
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as _driver:
        with _driver.session(database=os.getenv("NEO4J_DB")) as _session:
            return [r.data() for r in _session.run(query, params or {})]

# ---- 1) Build 3 lists ----
labels = sorted([r.get("label") for r in run_cypher("CALL db.labels()") if r.get("label") is not None])
predicates = sorted([r.get("relationshipType") for r in run_cypher("CALL db.relationshipTypes()") if r.get("relationshipType") is not None])

cities = [r["key"] for r in run_cypher("MATCH (c:City) RETURN c.key AS key")]
airports = [r["key"] for r in run_cypher("MATCH (a:Airport) RETURN a.key AS key")]
countries = [r["key"] for r in run_cypher("MATCH (c:Country) RETURN c.key AS key")]

doc_types = [r["v"] for r in run_cypher("MATCH (di:DocumentInstance) WHERE di.documentType IS NOT NULL RETURN DISTINCT di.documentType AS v")]
topic_keys = [r["v"] for r in run_cypher("MATCH (q:Questioning) WHERE q.topicKey IS NOT NULL RETURN DISTINCT q.topicKey AS v")]

keys = sorted(set(cities + airports + countries + doc_types + topic_keys))

# ---- 2) LLM selects filters ----

def llm_select(question):
    api_key = os.getenv("GPT_API_KEY")
    if not api_key:
        raise RuntimeError("GPT_API_KEY is not set")

    prompt = {
        "task": "Pick minimal filters from the provided lists to answer the question.",
        "rules": [
            "Output JSON only.",
            "Select only items from the provided lists.",
            "Document questions MUST include predicates: hasDocumentCheck, requestedDocument, documentType and labels: Encounter, DocumentCheck, DocumentInstance, Literal.",
            "If question mentions a city/airport, include atAirport, atCountry, locatedInCountry along with atCity/locatedInCity.",
            "If keys include ANY city/airport/country, you MUST include scope predicates: atCity, atAirport, atCountry, locatedInCity, locatedInCountry and labels: City, Airport, Country, Encounter.",
            "If keys include city_paris, you MUST also include keys: city_beauvais, airport_cdg, airport_ory, airport_bva.",
            "If you include hasDocumentCheck/requestedDocument, you MUST include Encounter, DocumentCheck, DocumentInstance labels.",
            "You MUST include atCity/atAirport/atCountry whenever scope keys are present.",
            "You MUST include documentType predicate for document questions.",
            "For document questions with a specific location (city/airport/country), you MUST include ALL documentType keys from the provided keys list (no subset).",
            "If you do not include ALL documentType keys, regenerate your JSON.",
            "If predicates do NOT include topicKey, you MUST NOT include any topic keys in keys.",
            "For document questions, include only documentType keys (plus scope keys); do NOT include topic keys unless question explicitly about questioning.",
"Keys must be ONLY scope keys (mentioned city/airport/country) and documentType keys for document questions. Do NOT include any other keys.",
            "Never include all cities/airports/countries; include only those explicitly mentioned or required by the rules (e.g., Paris special case).",
"Your output must be internally consistent: every key must be reachable by at least one predicate in your list.",
            "If any rule is violated, regenerate your JSON until all rules are satisfied.",
        ],
        "lists": {
            "labels": labels,
            "predicates": predicates,
            "keys": keys,
        },
        "output_format": {
            "labels": [],
            "predicates": [],
            "keys": []
        },
        "question": question
    }

    payload = {
        "model": LLM_MODEL,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": "Output JSON only."},
            {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)}
        ]
    }
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        data=json.dumps(payload),
    )
    if resp.status_code != 200:
        raise RuntimeError(f"OpenAI API error {resp.status_code}: {resp.text}")
    text = resp.json()["choices"][0]["message"]["content"].strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, re.S)
        return json.loads(m.group(0)) if m else {"labels": [], "predicates": [], "keys": []}

filters = llm_select(QUESTION)


print("FILTERS:", filters)

# ---- 3) Extract triples (depth<=2 + pseudo literals) ----

def extract_triples():
    triples = []

    q1 = """
    MATCH (e:Encounter)-[r]->(o)
    RETURN labels(e)[0] AS s_label, e.key AS s_key, type(r) AS p, labels(o)[0] AS o_label, o.key AS o_key
    """
    triples += run_cypher(q1)

    q2 = """
    MATCH (e:Encounter)-[]->(n)-[r]->(o)
    RETURN labels(n)[0] AS s_label, n.key AS s_key, type(r) AS p, labels(o)[0] AS o_label, o.key AS o_key
    """
    triples += run_cypher(q2)

    q3 = """
    MATCH (di:DocumentInstance) WHERE di.documentType IS NOT NULL
    RETURN 'DocumentInstance' AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di.documentType AS o_key
    """
    triples += run_cypher(q3)

    q4 = """
    MATCH (q:Questioning) WHERE q.topicKey IS NOT NULL
    RETURN 'Questioning' AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q.topicKey AS o_key
    """
    triples += run_cypher(q4)

    return triples


def normalize_triples(rows):
    out = []
    for r in rows:
        out.append({
            "s_label": r.get("s_label"),
            "s_key": r.get("s_key"),
            "p": r.get("p"),
            "o_label": r.get("o_label"),
            "o_key": r.get("o_key"),
        })
    return out

triples = normalize_triples(extract_triples())

# ---- 4) Filter by labels/predicates/keys ----

f_labels = set(filters.get("labels") or [])
f_preds = set(filters.get("predicates") or [])
f_keys = set(filters.get("keys") or [])

all_triples = list(triples)

if f_labels:
    triples = [t for t in triples if t.get("s_label") in f_labels or t.get("o_label") in f_labels]
if f_preds:
    triples = [t for t in triples if t.get("p") in f_preds]

# Scope-style filtering from keys (cities/airports/countries)
scope_keys = set(cities + airports) & f_keys  # only city/airport keys seed scope
if scope_keys:
    scope_preds = {"atCity", "atAirport", "atCountry", "locatedInCity", "locatedInCountry"}
    encounter_keys = {
        t.get("s_key")
        for t in all_triples
        if t.get("s_label") == "Encounter" and t.get("p") in scope_preds and t.get("o_key") in scope_keys
    }
    allowed = set(encounter_keys)
    changed = True
    while changed:
        changed = False
        for t in all_triples:
            s_key = t.get("s_key")
            o_key = t.get("o_key")
            if s_key in allowed and o_key not in allowed:
                allowed.add(o_key)
                changed = True
    triples = [t for t in triples if t.get("s_key") in allowed]

# Literal key filtering (documentType/topicKey) if present in keys
if f_keys:
    doc_keys = set(doc_types) & f_keys
    topic_sel = set(topic_keys) & f_keys
    if doc_keys:
        triples = [t for t in triples if not (t.get("p") == "documentType" and t.get("o_key") not in doc_keys)]
    if topic_sel:
        triples = [t for t in triples if not (t.get("p") == "topicKey" and t.get("o_key") not in topic_sel)]

# ---- 5) Metrics ----

def load_gold(qnum):
    import glob
    base = f"/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation/q_{qnum}"
    triples = []
    for path in glob.glob(base + "/**/annotation_*.json", recursive=True):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        triples.extend(data.get("triples", []))
    out = []
    for t in triples:
        out.append({
            "s_label": t["s"]["label"],
            "s_key": t["s"]["key"],
            "p": t["p"],
            "o_label": t["o"]["label"],
            "o_key": t["o"]["key"],
        })
    return out


def triples_to_set(triples):
    s = set()
    for t in triples:
        s.add((t.get("s_label"), t.get("s_key"), t.get("p"), t.get("o_label"), t.get("o_key")))
    return s

retrieved_set = triples_to_set(triples)
gold_set = triples_to_set(load_gold(Q_NUM))
inter = retrieved_set & gold_set
precision = len(inter) / len(retrieved_set) if retrieved_set else 0.0
recall = len(inter) / len(gold_set) if gold_set else 0.0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print(f"retrieved {len(retrieved_set)}")
print(f"gold {len(gold_set)}")
print(f"precision {precision}")
print(f"recall {recall}")
print(f"f1 {f1}")









FILTERS: {'labels': ['Airport', 'Country', 'Encounter'], 'predicates': ['atAirport', 'atCountry', 'locatedInCountry'], 'keys': ['country_germany']}
retrieved 736
gold 536
precision 0.4483695652173913
recall 0.6156716417910447
f1 0.5188679245283019


In [ ]:
# Save retrieved triples to file (JSON)
import json
from pathlib import Path

out_path = Path("/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/retrieved_triples_current.json")
out_path.write_text(json.dumps(triples, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved", out_path)
